# Solutions · Advanced: the real survey file

One worked answer per exercise. There is usually more than one correct way to write these: if your self-check passed, your answer is right too.

Read this **after** you've had a go. Seeing the answer teaches far less than getting stuck and working it out.

In [ ]:
# Run this first. It loads the survey file exactly as the researchers published it.
import urllib.request
from pathlib import Path

import pandas as pd

FILE = "pollinators.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

raw = pd.read_csv(FILE)
print("Loaded", FILE, raw.shape)

In [ ]:
# Make a working copy. Changes to df won't touch raw.
df = raw.copy()
df.info()

## Solution: Challenge A1: the column that won't work

In [ ]:
print(list(df.columns))               # the quotes show the space: 'Hortorum '
df.columns = df.columns.str.strip()   # strip spaces from every name at once
df["Hortorum"].sum()

## Solution: Challenge A2: missing values

In [ ]:
print(df.isna().sum())                 # only Plants has gaps: 41
missing = df[df["Plants"].isna()]
print(missing["Crop_Generic"].value_counts())
most_missing = missing["Crop_Generic"].value_counts().index[0]

# Check yourself: dropna() would remove all 41 rows
print(len(df) - len(df.dropna()), "surveys would be lost by dropna()")

## Solution: Challenge A3: rows that appear twice

In [ ]:
n_repeats = df.duplicated().sum()
print(n_repeats, "repeated rows")

# Check yourself: removing them pushes the average up, because almost all of them are zeros
print(round(df["Bumblebees"].mean(), 3), "with them,", round(df.drop_duplicates()["Bumblebees"].mean(), 3), "without")
df[df.duplicated(keep=False)].head(10)

## Solution: Challenge A4: dates that are really text

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
mismatch = df[df["Date"].dt.month_name() != df["Month"]]
mismatch[["Allotment", "Date", "Month"]]

In [ ]:
# Run this to fix the six labels, and give two columns the same names as in the tidy file
df["Month"] = df["Date"].dt.month_name()
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
print((df["Date"].dt.month_name() != df["Month"]).sum(), "mismatches left")

## Solution: Challenge B1: do shared plots attract more insects?

In [ ]:
insects = ["Beetles", "Hoverflies", "Flies", "Butterflies", "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]
df["All_insects"] = df[insects].sum(axis=1)

plots = df.groupby("Community").agg(
    surveys=("All_insects", "size"),
    insects=("All_insects", "sum"),
    flowers=("Flowers", "sum"),
)
plots["per_survey"] = plots["insects"] / plots["surveys"]
plots["per_flower"] = plots["insects"] / plots["flowers"]
plots.round(4)

## Solution: Challenge B2: which bumblebees visit broad bean?

In [ ]:
sp = ["Terrestris", "Lapidarius", "Pascuorum", "Pratorum", "Hypnorum", "Hortorum"]

species = df.groupby("Crop")[sp].sum()
species = species[species.sum(axis=1) >= 20]     # crops with at least 20 bumblebee visits
species_pct = (species.div(species.sum(axis=1), axis=0) * 100).round()
species_pct

## Solution: Challenge B3: was 2018 a worse year?

In [ ]:
years = df.groupby("Year").agg(
    surveys=("Bumblebees", "size"),
    bumblebees=("Bumblebees", "sum"),
    honeybees=("Honeybees", "sum"),
)
years["bb_per_survey"] = years["bumblebees"] / years["surveys"]
years["hb_per_survey"] = years["honeybees"] / years["surveys"]
bb_2018 = round(years.loc[2018, "bb_per_survey"], 2)
print(years.round(2))

# Are the two years comparable? 2017 has no April surveys at all.
df.groupby(["Year", "Month"]).size()

## Your own question: one example

In [ ]:
# Do the bumblebee species peak in different months?
order = ["April", "May", "June", "July", "August", "September"]
by_month = df.groupby("Month")[sp].sum().loc[order]
by_month
# B. pratorum (the early bumblebee) is busiest in May and June and almost gone by August.
# B. terrestris peaks in June. B. pascuorum keeps building to a big peak in August (133 visits),
# and is the only species still seen in September. Before trusting this, check how many
# surveys each month had (August had the most):
# df["Month"].value_counts()